In [1]:
!conda env export > environment.yml
!pip install transformers datasets scikit-learn
!pip install torch torchvision torchaudio
!pip install tensorflow
!pip install tensorflow
!pip install hf_xet
!pip install tqdm
!pip install evaluate
!pip install "accelerate>=0.26.0"
!pip install ipywidgets
!pip install huggingface_hub

In [14]:
import pandas as pd
from bs4 import BeautifulSoup
import re
import nltk
from nltk.corpus import stopwords
import numpy as np
import transformers
import numpy as np
from transformers import pipeline
import torch
import tensorflow
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import (
    BartForSequenceClassification,
    BartTokenizerFast,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from datasets import Dataset, DatasetDict
import evaluate 
import transformers 

In [15]:
# Function to extract text from HTML
def extract_text(html):
    soup = BeautifulSoup(html, "html.parser")
    return soup.get_text(separator=" ", strip=True)

# Load the csv file
# df = pd.read_csv('D:/comp-6713-industry-project/raw_data/seniority_labelled_development_set.csv')
# df_test = pd.read_csv('D:/comp-6713-industry-project/raw_data/seniority_labelled_test_set.csv')
df = pd.read_csv('raw_data/seniority_labelled_development_set.csv')
df_test = pd.read_csv('raw_data/seniority_labelled_test_set.csv')


In [16]:
apostrophe_dict = {
"ain't": "am not / are not",
"aren't": "are not / am not",
"can't": "cannot",
"can't've": "cannot have",
"'cause": "because",
"could've": "could have",
"couldn't": "could not",
"couldn't've": "could not have",
"didn't": "did not",
"doesn't": "does not",
"don't": "do not",
"hadn't": "had not",
"hadn't've": "had not have",
"hasn't": "has not",
"haven't": "have not",
"he'd": "he had / he would",
"he'd've": "he would have",
"he'll": "he shall / he will",
"he'll've": "he shall have / he will have",
"he's": "he has / he is",
"how'd": "how did",
"how'd'y": "how do you",
"how'll": "how will",
"how's": "how has / how is",
"i'd": "I had / I would",
"i'd've": "I would have",
"i'll": "I shall / I will",
"i'll've": "I shall have / I will have",
"i'm": "I am",
"i've": "I have",
"isn't": "is not",
"it'd": "it had / it would",
"it'd've": "it would have",
"it'll": "it shall / it will",
"it'll've": "it shall have / it will have",
"it's": "it has / it is",
"let's": "let us",
"ma'am": "madam",
"mayn't": "may not",
"might've": "might have",
"mightn't": "might not",
"mightn't've": "might not have",
"must've": "must have",
"mustn't": "must not",
"mustn't've": "must not have",
"needn't": "need not",
"needn't've": "need not have",
"o'clock": "of the clock",
"oughtn't": "ought not",
"oughtn't've": "ought not have",
"shan't": "shall not",
"sha'n't": "shall not",
"shan't've": "shall not have",
"she'd": "she had / she would",
"she'd've": "she would have",
"she'll": "she shall / she will",
"she'll've": "she shall have / she will have",
"she's": "she has / she is",
"should've": "should have",
"shouldn't": "should not",
"shouldn't've": "should not have",
"so've": "so have",
"so's": "so as / so is",
"that'd": "that would / that had",
"that'd've": "that would have",
"that's": "that has / that is",
"there'd": "there had / there would",
"there'd've": "there would have",
"there's": "there has / there is",
"they'd": "they had / they would",
"they'd've": "they would have",
"they'll": "they shall / they will",
"they'll've": "they shall have / they will have",
"they're": "they are",
"they've": "they have",
"to've": "to have",
"wasn't": "was not",
"we'd": "we had / we would",
"we'd've": "we would have",
"we'll": "we will",
"we'll've": "we will have",
"we're": "we are",
"we've": "we have",
"weren't": "were not",
"what'll": "what shall / what will",
"what'll've": "what shall have / what will have",
"what're": "what are",
"what's": "what has / what is",
"what've": "what have",
"when's": "when has / when is",
"when've": "when have",
"where'd": "where did",
"where's": "where has / where is",
"where've": "where have",
"who'll": "who shall / who will",
"who'll've": "who shall have / who will have",
"who's": "who has / who is",
"who've": "who have",
"why's": "why has / why is",
"why've": "why have",
"will've": "will have",
"won't": "will not",
"won't've": "will not have",
"would've": "would have",
"wouldn't": "would not",
"wouldn't've": "would not have",
"y'all": "you all",
"y'all'd": "you all would",
"y'all'd've": "you all would have",
"y'all're": "you all are",
"y'all've": "you all have",
"you'd": "you had / you would",
"you'd've": "you would have",
"you'll": "you shall / you will",
"you'll've": "you shall have / you will have",
"you're": "you are",
"you've": "you have"
}

In [17]:
nltk.download('stopwords') 
nltk.download('punkt_tab')

In [18]:
# Function to expand contractions using regex for word boundaries
def expand_apostrophe(text, apostrophe_dict):
    pattern = re.compile(r'\b(' + '|'.join(map(re.escape, apostrophe_dict.keys())) + r')\b')
    return pattern.sub(lambda match: apostrophe_dict[match.group(0)], text)

In [19]:
from nltk.tokenize import word_tokenize

# Remove stopwords and punctuation from input text
stop_words = set(stopwords.words('english'))
def remove_stopwords(text):
    if not isinstance(text, str):
        return text
    text = re.sub(r'[^\w\s]', ' ', text)
    tokens = word_tokenize(text)
    filtered = [t for t in tokens if t not in stop_words and t.strip()]
    return ' '.join(filtered)

In [20]:
def clean_text(text):
    # Replace non-breaking spaces (\xa0) with a normal space
    text = text.replace("\xa0", " ")
    
    # Remove specific punctuation characters: +, /, @, -
    # You can modify this regex pattern to include other characters if needed.
    text = re.sub(r"[+/@]", "", text)

    # Optionally, you might remove all punctuation:
    # text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [21]:
job_details = (
    df['job_ad_details']
      .apply(extract_text)
      .str.lower()
      .str.replace(r'\bjob title\b', '', regex=True) 
      .apply(lambda x: expand_apostrophe(x, apostrophe_dict))
      .apply(remove_stopwords)
      .apply(clean_text)
)
job_test_details = (
    df_test['job_ad_details']
      .apply(extract_text)
      .str.lower()
      .str.replace(r'\bjob title\b', '', regex=True) 
      .apply(lambda x: expand_apostrophe(x, apostrophe_dict))
      .apply(remove_stopwords)
      .apply(clean_text)
)

In [22]:
label_mapping = {
    "entry level": "entry", "junior": "entry", "graduate": "entry",
    "trainee": "entry", "student": "entry", "entry-level": "entry",
    "entry level assistant": "entry", "1st year apprentice": "entry",
    "2nd year apprentice": "entry", "apprentice": "entry",

    "intermediate": "intermediate", "mid-level": "intermediate",
    "associate": "intermediate", "assistant": "intermediate",
    "standard": "intermediate", "coordinator": "intermediate",
    "experienced": "intermediate", "qualified": "intermediate",
    "junior-intermediate": "intermediate", "experienced assistant": "intermediate",

    "senior": "senior", "lead": "senior", "senior associate": "senior",
    "senior lead": "senior", "senior/lead": "senior", "senior-executive": "senior",
    "senior assistant": "senior", "senior head": "senior",
    "mid-senior": "senior", "intermediate to senior": "senior",

    "head": "management", "director": "management", "assistant manager": "management",
    "assistant head": "management", "assistant director": "management",
    "associate director": "management", "regional head": "management",
    "middle-management": "management", "manager": "management",

    "executive": "executive", "chief": "executive", "principal": "executive",
    "general-manager": "executive", "owner": "executive", "owner-operator": "executive",
    "board": "executive", "supervisor": "executive", "second-in-command": "executive",
    "independent": "executive", "advanced": "executive",
}

In [23]:
# Map original labels to new labels using label_mapping dictionary
mapped_label = df['y_true'].apply(lambda x: label_mapping.get(x, 'other'))
mapped_label_test = df_test['y_true'].apply(lambda x: label_mapping.get(x, 'other'))

# Get unique labels and their counts from training data
unique_labels = mapped_label.unique().tolist()
label_counts = mapped_label.value_counts().to_dict()
print("Unique labels:", unique_labels)
print("Number of unique labels:", len(unique_labels))
print(label_counts)

Unique labels: ['intermediate', 'senior', 'management', 'entry', 'executive', 'other']
Number of unique labels: 6
{'intermediate': 1599, 'senior': 583, 'entry': 380, 'management': 103, 'executive': 55, 'other': 32}


In [24]:
from sklearn.metrics import f1_score, classification_report

predictions = []
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0)
# Iterate over the job advertisement texts
for job_ad in job_details:
    result = classifier(job_ad, unique_labels)
    # The classifier returns a dictionary with keys "labels" and "scores".
    # 'labels' is sorted from highest to lowest score, so we take the first one.
    pred_label = result["labels"][0]
    predictions.append(pred_label)

# Optionally, you can store the predictions in the dataframe for further analysis.
df['predicted_seniority'] = predictions

# Step 4: Compute the F1 Score
# Using scikit-learn's f1_score function. Choose an averaging method (e.g., 'weighted', 'macro').
# Make sure that df_subset['y_true'] contains the correct ground-truth labels.
f1 = f1_score(mapped_label, predictions, average='weighted')
print("Weighted F1 Score:", f1)

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Weighted F1 Score: 0.2616426696302603


In [25]:
from sklearn.metrics import classification_report
print(classification_report(mapped_label, predictions))

              precision    recall  f1-score   support

       entry       0.34      0.26      0.29       380
   executive       0.15      0.27      0.19        55
intermediate       0.71      0.12      0.20      1599
  management       0.08      0.57      0.13       103
       other       0.01      0.22      0.02        32
      senior       0.47      0.43      0.45       583

    accuracy                           0.22      2752
   macro avg       0.29      0.31      0.21      2752
weighted avg       0.56      0.22      0.26      2752



In [27]:
job_ad = job_details.tolist()
job_ad_test = job_test_details.tolist()
labels = mapped_label.tolist()
labels_test = mapped_label_test.tolist()

le = LabelEncoder()
label_ids = le.fit_transform(labels)
label_ids_test = le.fit_transform(labels_test)
num_labels = len(le.classes_)

In [28]:
# Split data into training/validation sets with stratified sampling
train_texts, val_texts, train_labels, val_labels = train_test_split(
    job_ad, label_ids,
    test_size=0.3,
    random_state=42,
    stratify=label_ids
)

train_dataset = Dataset.from_dict({'text': train_texts, 'labels': train_labels})
val_dataset   = Dataset.from_dict({'text': val_texts,   'labels': val_labels})
test_dataset = Dataset.from_dict({'text': job_ad_test,   'labels': label_ids_test})
datasets = DatasetDict({'train': train_dataset, 'validation': val_dataset})

In [29]:
tokenizer = BartTokenizerFast.from_pretrained('facebook/bart-large-mnli')

# Define batch tokenization function for text processing
def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, max_length=512)

tokenized = datasets.map(tokenize_fn, batched=True)
tokenized.set_format(type='torch', columns=['input_ids','attention_mask','labels'])
print(tokenized)

# Process test set with same tokenization
tokenized_test = test_dataset.map(tokenize_fn, batched=True)
tokenized_test.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

Map:   0%|          | 0/1926 [00:00<?, ? examples/s]

Map:   0%|          | 0/826 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 1926
    })
    validation: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 826
    })
})


Map:   0%|          | 0/689 [00:00<?, ? examples/s]

In [30]:
# Initialize BART model for sequence classification task
model = BartForSequenceClassification.from_pretrained(
    'facebook/bart-large-mnli',
    num_labels=6,
    ignore_mismatched_sizes=True
)
data_collator = DataCollatorWithPadding(tokenizer)

Some weights of BartForSequenceClassification were not initialized from the model checkpoint at facebook/bart-large-mnli and are newly initialized because the shapes did not match:
- classification_head.out_proj.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([6]) in the model instantiated
- classification_head.out_proj.weight: found shape torch.Size([3, 1024]) in the checkpoint and torch.Size([6, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [31]:
# Compute weighted F1 score for model evaluation
def compute_metrics(eval_pred):
    logits = eval_pred.predictions
    if isinstance(logits, tuple):
        logits = logits[0]  # unwrap logits
    preds = np.argmax(logits, axis=1)
    labels = eval_pred.label_ids
    return {"weighted_f1": f1_score(labels, preds, average="weighted")}

In [32]:
import torch
from torch.utils.data import DataLoader
from transformers.optimization import get_linear_schedule_with_warmup
from torch.optim import AdamW
from tqdm import tqdm

# 1) Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Using device:", device)

# 2) DataLoaders
train_loader = DataLoader(tokenized['train'],
                          batch_size=8,
                          shuffle=True,
                          collate_fn=data_collator)
val_loader = DataLoader(tokenized['validation'],
                        batch_size=8,
                        collate_fn=data_collator)

# 3) Optimizer & Scheduler
optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 10
total_steps = len(train_loader) * num_epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

best_model_path = "./best_model"
best_val_loss = float("inf")

for epoch in range(1, num_epochs + 1):
    # Training loop (same as before)
    model.train()
    total_train_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Train Epoch {epoch}"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_train_loss += loss.item()
    avg_train_loss = total_train_loss / len(train_loader)

    # Validation loop
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            total_val_loss += outputs.loss.item()
    avg_val_loss = total_val_loss / len(val_loader)

    print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        model.save_pretrained(best_model_path)
        tokenizer.save_pretrained(best_model_path)
        print(f"New best model saved at epoch {epoch}")
    model.save_pretrained("./final_model")
    tokenizer.save_pretrained("./final_model")

Train Epoch 1: 100%|█████████████████████████████████████████████████████████████████| 241/241 [01:50<00:00,  2.18it/s]


Epoch 1 | Train Loss: 1.1215 | Val Loss: 0.9089
✅ New best model saved at epoch 1


Train Epoch 2: 100%|█████████████████████████████████████████████████████████████████| 241/241 [01:49<00:00,  2.21it/s]


Epoch 2 | Train Loss: 0.8001 | Val Loss: 0.8256
✅ New best model saved at epoch 2


Train Epoch 3: 100%|█████████████████████████████████████████████████████████████████| 241/241 [01:49<00:00,  2.19it/s]


...

Epoch 8 | Train Loss: 0.0248 | Val Loss: 1.4114


Train Epoch 9: 100%|█████████████████████████████████████████████████████████████████| 241/241 [01:51<00:00,  2.17it/s]


Epoch 9 | Train Loss: 0.0128 | Val Loss: 1.4657


Train Epoch 10: 100%|████████████████████████████████████████████████████████████████| 241/241 [01:49<00:00,  2.20it/s]


Epoch 10 | Train Loss: 0.0101 | Val Loss: 1.4845


In [33]:
from sklearn.metrics import accuracy_score

def evaluate_accuracy(model, dataloader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    print(f"Accuracy: {acc:.4f}")
    return acc

In [34]:
test_loader = DataLoader(
    tokenized_test,
    batch_size=16,           
    collate_fn=data_collator
)

In [35]:
# Reload the best model
model = BartForSequenceClassification.from_pretrained(best_model_path).to(device)

# Evaluate on validation set (or replace with test_loader if you have test data)
evaluate_accuracy(model, test_loader)

You passed along `num_labels=3` with an incompatible id to label map: {'0': 'LABEL_0', '1': 'LABEL_1', '2': 'LABEL_2', '3': 'LABEL_3', '4': 'LABEL_4', '5': 'LABEL_5'}. The number of labels wil be overwritten to 6.
Evaluating: 100%|██████████████████████████████████████████████████████████████████████| 87/87 [00:12<00:00,  6.88it/s]

✅ Accuracy: 0.6778


0.6777939042089985

In [36]:
# Reload the best model
model = BartForSequenceClassification.from_pretrained("./final_model").to(device)

# Evaluate on validation set (or replace with test_loader if you have test data)
evaluate_accuracy(model, test_loader)

You passed along `num_labels=3` with an incompatible id to label map: {'0': 'LABEL_0', '1': 'LABEL_1', '2': 'LABEL_2', '3': 'LABEL_3', '4': 'LABEL_4', '5': 'LABEL_5'}. The number of labels wil be overwritten to 6.
Evaluating: 100%|██████████████████████████████████████████████████████████████████████| 87/87 [00:12<00:00,  6.87it/s]

✅ Accuracy: 0.6967


0.6966618287373004

In [37]:
from sklearn.metrics import classification_report

def get_predictions_and_labels(model, dataloader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"]
            )
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch["labels"].cpu().numpy())
    return all_preds, all_labels

In [38]:
best_model = BartForSequenceClassification.from_pretrained(best_model_path).to(device)
best_preds, best_labels = get_predictions_and_labels(best_model, test_loader)

print("Classification Report (Best Model):")
print(classification_report(best_labels, best_preds, target_names=le.classes_))

You passed along `num_labels=3` with an incompatible id to label map: {'0': 'LABEL_0', '1': 'LABEL_1', '2': 'LABEL_2', '3': 'LABEL_3', '4': 'LABEL_4', '5': 'LABEL_5'}. The number of labels wil be overwritten to 6.


📊 Classification Report (Best Model):
              precision    recall  f1-score   support

       entry       0.62      0.55      0.58       111
   executive       0.00      0.00      0.00        17
intermediate       0.75      0.81      0.78       386
  management       0.25      0.04      0.06        27
       other       0.00      0.00      0.00        15
      senior       0.54      0.68      0.60       133

    accuracy                           0.68       689
   macro avg       0.36      0.35      0.34       689
weighted avg       0.64      0.68      0.65       689



C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
